# Beach Human Detection — Faster R-CNN + Four Preprocessing Methods

**Configuration:** Batch size **10**, **100 epochs**, Faster R-CNN ResNet-50 FPN v2, and persistent checkpoints saved to Google Drive after every epoch.

**Classes used throughout the notebook**
- `0 = Human on Land`
- `1 = Human in Water`

**Visualization style:** neon-green labels with transparent backgrounds. Ground-truth annotations and model predictions are drawn separately so missed detections can be spotted.

> Important: no object detector can guarantee that it will never miss a bounding box. This notebook draws every ground-truth box present in the annotation file and every prediction above the chosen confidence threshold. Recall and other metrics should be used to assess missed detections.


## 1. Install dependencies

In [ ]:
!pip -q install roboflow opencv-python-headless scikit-learn seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.2 MB/s eta 0:00:00


## 2. Mount Google Drive and configure persistent output

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, random, json, time, copy, math, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS = 100
BATCH_SIZE = 10
NUM_WORKERS = 2
LEARNING_RATE = 0.005
CONFIDENCE_THRESHOLD = 0.30

# This is a NEW experiment folder; it does not overwrite earlier project checkpoints.
OUT = Path("/content/drive/MyDrive/BHD_batch10_preprocessing_experiment")
OUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Batch size:", BATCH_SIZE)
print("Epochs:", NUM_EPOCHS)
print("Checkpoints:", CHECKPOINT_DIR)


ValueError: Mountpoint must not already contain files

## 3. Download your Roboflow dataset

In [ ]:
from getpass import getpass
from roboflow import Roboflow

# Enter a NEW/rotated Roboflow API key when prompted; avoid sharing it in notebook output.
ROBOFLOW_API_KEY = getpass("Roboflow API key: ")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("image-analysis-project-rfnyq").project("bhd-m9qqy")
version = project.version(1)
dataset = version.download("yolov8")

DATASET_DIR = Path(dataset.location)
print("Dataset location:", DATASET_DIR)
print("data.yaml exists:", (DATASET_DIR / "data.yaml").exists())


Roboflow API key: ··········
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to bhd-1 in yolov8:: 100%|██████████| 479/479 [00:00<00:00, 6268.12it/s]

Dataset location: /content/bhd-1
data.yaml exists: True


## 4. Inspect dataset class names before training

In [ ]:
import yaml

yaml_path = DATASET_DIR / "data.yaml"
with open(yaml_path, "r") as f:
    CFG = yaml.safe_load(f)

raw_names = CFG.get("names", {})
if isinstance(raw_names, list):
    names = {i: str(v) for i, v in enumerate(raw_names)}
else:
    names = {int(k): str(v) for k, v in raw_names.items()}

print("Classes declared in data.yaml:")
for k, v in sorted(names.items()):
    print(f"  Source ID {k}: {v}")

print("\nIMPORTANT: verify class IDs against Roboflow's class order/annotated samples.")
print("This notebook assumes source ID 0 = Human on Land and source ID 1 = Human in Water.")
print("If your Roboflow project uses the reverse order, change SOURCE_LAND_ID and SOURCE_WATER_ID below.")


Classes declared in data.yaml:
  Source ID 0: 0
  Source ID 1: 1

IMPORTANT: verify class IDs against Roboflow's class order/annotated samples.
This notebook assumes source ID 0 = Human on Land and source ID 1 = Human in Water.
If your Roboflow project uses the reverse order, change SOURCE_LAND_ID and SOURCE_WATER_ID below.


## 5. Configure source-to-semantic mapping

In [ ]:
# Verify this mapping using your actual annotation/class order.
SOURCE_LAND_ID = 0
SOURCE_WATER_ID = 1

if SOURCE_LAND_ID == SOURCE_WATER_ID:
    raise ValueError("Land and water source IDs must be different.")
if SOURCE_LAND_ID not in names or SOURCE_WATER_ID not in names:
    raise ValueError(f"Expected source IDs not found in data.yaml. Found: {names}")

SOURCE_TO_SEMANTIC = {
    SOURCE_LAND_ID: 0,  # semantic 0 = Human on Land
    SOURCE_WATER_ID: 1, # semantic 1 = Human in Water
}
SEMANTIC_NAMES = {0: "Human on Land", 1: "Human in Water"}

print("Source mapping:", SOURCE_TO_SEMANTIC)
print("Display labels:", SEMANTIC_NAMES)


Source mapping: {0: 0, 1: 1}
Display labels: {0: 'Human on Land', 1: 'Human in Water'}


## 6. Find train/valid/test folders

In [ ]:
def find_split(name):
    candidates = [
        DATASET_DIR / name / "images",
        DATASET_DIR / name,
    ]
    for p in candidates:
        if p.exists() and p.is_dir():
            # If direct folder contains images, use it.
            if any(x.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"} for x in p.iterdir() if x.is_file()):
                return p
    return None

TRAIN_DIR = find_split("train")
VAL_DIR = find_split("valid") or find_split("val")
TEST_DIR = find_split("test")

if TRAIN_DIR is None:
    raise FileNotFoundError(f"Could not find train images under {DATASET_DIR}")
if VAL_DIR is None:
    print("WARNING: no validation split found; validation loss will be skipped.")
if TEST_DIR is None:
    print("WARNING: no test split found; final test visualization will be skipped.")

print("Train:", TRAIN_DIR)
print("Validation:", VAL_DIR)
print("Test:", TEST_DIR)


Train: /content/bhd-1/train/images
Validation: /content/bhd-1/valid/images
Test: /content/bhd-1/test/images


## 7. Dataset loader with randomized preprocessing during training

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
METHODS = ["clahe", "gaussian", "median", "bilateral"]

def paired_label_path(image_path, split_dir):
    # Standard Roboflow YOLO layout: split/images/file.jpg and split/labels/file.txt
    if image_path.parent.name == "images":
        label_dir = image_path.parent.parent / "labels"
    else:
        label_dir = split_dir / "labels"
    return label_dir / f"{image_path.stem}.txt"

def preprocess_rgb(rgb, method):
    if method == "original":
        return rgb
    if method == "clahe":
        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
        return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
    if method == "gaussian":
        return cv2.GaussianBlur(rgb, (5, 5), 0)
    if method == "median":
        return cv2.medianBlur(rgb, 5)
    if method == "bilateral":
        return cv2.bilateralFilter(rgb, 9, 75, 75)
    raise ValueError(f"Unknown preprocessing method: {method}")

class BHDDataset(Dataset):
    def __init__(self, split_dir, training=False, method="original"):
        self.split_dir = Path(split_dir)
        self.training = training
        self.method = method
        image_root = self.split_dir
        if (self.split_dir / "images").exists():
            image_root = self.split_dir / "images"
        self.images = sorted(p for p in image_root.iterdir()
                             if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
        if not self.images:
            raise FileNotFoundError(f"No images found in {image_root}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        path = self.images[idx]
        rgb = np.array(Image.open(path).convert("RGB"))
        method = random.choice(METHODS) if self.training else self.method
        rgb = preprocess_rgb(rgb, method)
        h, w = rgb.shape[:2]

        label_path = paired_label_path(path, self.split_dir)
        boxes, labels = [], []
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                source_id = int(float(parts[0]))
                if source_id not in SOURCE_TO_SEMANTIC:
                    continue
                xc, yc, bw, bh = map(float, parts[1:5])
                x1 = max(0.0, (xc - bw / 2) * w)
                y1 = max(0.0, (yc - bh / 2) * h)
                x2 = min(float(w), (xc + bw / 2) * w)
                y2 = min(float(h), (yc + bh / 2) * h)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    # Torchvision label 0 is background; foreground labels are 1 and 2.
                    labels.append(SOURCE_TO_SEMANTIC[source_id] + 1)

        img = torch.from_numpy(rgb.copy()).permute(2, 0, 1).float() / 255.0
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx], dtype=torch.int64),
            "area": torch.tensor([(b[2]-b[0])*(b[3]-b[1]) for b in boxes], dtype=torch.float32),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
            "path": str(path),
        }
        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

train_ds = BHDDataset(TRAIN_DIR, training=True)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=collate_fn,
    pin_memory=(DEVICE.type == "cuda")
)
val_loader = None
if VAL_DIR is not None:
    val_ds = BHDDataset(VAL_DIR, training=False, method="original")
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, collate_fn=collate_fn,
        pin_memory=(DEVICE.type == "cuda")
    )

print("Training images:", len(train_ds))
print("Batches per epoch:", len(train_loader))
print("Batch size:", BATCH_SIZE)


Training images: 166
Batches per epoch: 17
Batch size: 10


## 8. Preview dataset annotations

In [ ]:
# Preview one image with every annotation box.
preview_img, preview_target = train_ds[0]
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(preview_img.permute(1, 2, 0).numpy())
for box, label in zip(preview_target["boxes"].numpy(), preview_target["labels"].numpy()):
    x1, y1, x2, y2 = box
    semantic_id = int(label) - 1
    ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                               edgecolor="#39FF14", linewidth=2))
    ax.text(x1, max(0, y1-4), f"{semantic_id} {SEMANTIC_NAMES[semantic_id]}",
            color="#39FF14", fontsize=9,
            bbox=dict(facecolor="none", edgecolor="none", alpha=0))
ax.set_title("Ground truth annotations — neon green")
ax.axis("off")
plt.show()


NameError: name 'train_ds' is not defined

## 9. Build Faster R-CNN model

In [ ]:
# Uses COCO pretrained weights for initialization; this is a new training run.
model = fasterrcnn_resnet50_fpn_v2(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 3)  # background + 2 classes
model = model.to(DEVICE)

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, momentum=0.9, weight_decay=0.0005
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

print("Model ready on", DEVICE)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:01<00:00, 166MB/s]


Model ready on cuda


In [ ]:
from google.colab import drive
import os

MOUNT_PATH = "/content/gdrive"

os.makedirs(MOUNT_PATH, exist_ok=True)

drive.mount(MOUNT_PATH)

Mounted at /content/gdrive


In [ ]:
from pathlib import Path

drive_root = Path("/content/gdrive/MyDrive")

print("Drive accessible:", drive_root.exists())

for name in ("fasterrcnn_last.pth", "fasterrcnn_best.pth"):
    matches = list(drive_root.rglob(name))
    if matches:
        for p in matches:
            print(f"FOUND: {p} ({p.stat().st_size / (1024**2):.2f} MB)")
    else:
        print("NOT FOUND:", name)

Drive accessible: True
FOUND: /content/gdrive/MyDrive/beach_human_detection_outputs/fasterrcnn_last.pth (329.63 MB)
FOUND: /content/gdrive/MyDrive/beach_human_detection_corrected/fasterrcnn_last.pth (329.63 MB)
FOUND: /content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_last.pth (329.62 MB)
FOUND: /content/gdrive/MyDrive/beach_human_detection_outputs/fasterrcnn_best.pth (329.62 MB)
FOUND: /content/gdrive/MyDrive/beach_human_detection_corrected/fasterrcnn_best.pth (329.62 MB)
FOUND: /content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_best.pth (165.39 MB)


In [ ]:
import torch
from pathlib import Path

LATEST_CKPT = Path(
    "/content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_last.pth"
)

ckpt = torch.load(
    LATEST_CKPT,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint loaded successfully")
print("Last completed epoch:", ckpt["epoch"] + 1)
print("Next epoch:", ckpt["epoch"] + 2)
print("Training history entries:", len(ckpt["history"]["train_loss"]))
print("Batch size saved:", ckpt.get("batch_size", "Not recorded"))

Checkpoint loaded successfully
Last completed epoch: 88
Next epoch: 89
Training history entries: 88
Batch size saved: 10


In [ ]:
from pathlib import Path

OUT = Path("/content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment")
CHECKPOINT_DIR = OUT / "checkpoints"

LATEST_CKPT = CHECKPOINT_DIR / "fasterrcnn_last.pth"
BEST_CKPT = CHECKPOINT_DIR / "fasterrcnn_best.pth"
HISTORY_PATH = OUT / "training_history.json"

RESUME = True

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. Enable T4 GPU in Runtime settings.")

Device: cpu


## 10. Checkpointing and resume support

In [ ]:
LATEST_CKPT = CHECKPOINT_DIR / "fasterrcnn_last.pth"
BEST_CKPT = CHECKPOINT_DIR / "fasterrcnn_best.pth"
HISTORY_PATH = OUT / "training_history.json"

start_epoch = 0
best_val = float("inf")
history = {"train_loss": [], "val_loss": []}

# If you want to resume this NEW experiment after a runtime reset, set RESUME=True.
RESUME = True

if RESUME and LATEST_CKPT.exists():
    checkpoint = torch.load(LATEST_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    start_epoch = int(checkpoint["epoch"]) + 1
    best_val = float(checkpoint.get("best_val", best_val))
    history = checkpoint.get("history", history)
    print(f"Resuming from epoch {start_epoch + 1}")
else:
    print("Starting a fresh experiment. Existing project checkpoints will not be loaded.")


NameError: name 'model' is not defined

## 11. Train for 100 epochs and save after every epoch

In [ ]:
def run_train_epoch(model, loader, optimizer, epoch):
    model.train()
    running = 0.0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for images, targets in pbar:
        images = [im.to(DEVICE, non_blocking=True) for im in images]
        targets_device = []
        for t in targets:
            targets_device.append({
                k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v)
                for k, v in t.items() if k != "path"
            })
        loss_dict = model(images, targets_device)
        loss = sum(loss_dict.values())
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at epoch {epoch+1}: {loss_dict}")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        running += float(loss.detach().cpu())
        pbar.set_postfix(loss=f"{float(loss.detach().cpu()):.4f}")
    return running / max(1, len(loader))

@torch.no_grad()
def calculate_val_loss(model, loader):
    if loader is None:
        return float("nan")
    # Torchvision detection models return losses in train mode; disable gradients.
    was_training = model.training
    model.train()
    total, count = 0.0, 0
    for images, targets in tqdm(loader, desc="Validation", leave=False):
        images = [im.to(DEVICE, non_blocking=True) for im in images]
        targets_device = [{
            k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v)
            for k, v in t.items() if k != "path"
        } for t in targets]
        loss_dict = model(images, targets_device)
        loss = sum(loss_dict.values())
        if torch.isfinite(loss):
            total += float(loss.cpu())
            count += 1
    model.train(was_training)
    return total / max(1, count) if count else float("nan")

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()
    train_loss = run_train_epoch(model, train_loader, optimizer, epoch)
    val_loss = calculate_val_loss(model, val_loader)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if math.isfinite(val_loss) and val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), BEST_CKPT)

    # Save full recovery checkpoint after EVERY epoch.
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val": best_val,
        "history": history,
        "class_names": SEMANTIC_NAMES,
        "source_to_semantic": SOURCE_TO_SEMANTIC,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
    }
    torch.save(checkpoint, LATEST_CKPT)
    HISTORY_PATH.write_text(json.dumps(history, indent=2))

    print(
        f"Epoch {epoch+1:03d}/{NUM_EPOCHS}: "
        f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
        f"time={(time.time()-epoch_start)/60:.1f} min"
    )
    print("Saved recovery checkpoint:", LATEST_CKPT)

print("Training finished.")
print("Latest checkpoint:", LATEST_CKPT)
print("Best model weights:", BEST_CKPT if BEST_CKPT.exists() else "Not created")


Epoch 1/100:   0%|          | 0/17 [00:00<?, ?it/s]

Validation:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 001/100: train_loss=1.6789, val_loss=1.1896, time=1.0 min
Saved recovery checkpoint: /content/drive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_last.pth


Epoch 2/100:   0%|          | 0/17 [00:00<?, ?it/s]

Validation:   0%|          | 0/5 [00:00<?, ?it/s]

...

Epoch 87/100:   0%|          | 0/17 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dd62e46c0e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
Exception ignored in:     if w.is_alive():<function _MultiProcessingDataLoaderIter.__del__ at 0x7dd62e46c0e0>

  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
AssertionError    : self._shutdown_workers()
can only test a child process
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Validation:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 087/100: train_loss=0.3396, val_loss=0.5615, time=1.2 min
Saved recovery checkpoint: /content/drive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_last.pth


Epoch 88/100:   0%|          | 0/17 [00:00<?, ?it/s]

## 12. Plot training and validation loss

In [ ]:
if HISTORY_PATH.exists():
    hist = json.loads(HISTORY_PATH.read_text())
    plt.figure(figsize=(9, 5))
    plt.plot(range(1, len(hist["train_loss"])+1), hist["train_loss"], label="Train loss")
    valid_points = [(i+1, v) for i, v in enumerate(hist["val_loss"]) if math.isfinite(v)]
    if valid_points:
        plt.plot([x for x, _ in valid_points], [y for _, y in valid_points], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Faster R-CNN training history")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    print("No history found yet.")


## 13. Load best checkpoint for evaluation

In [ ]:
# Load the best saved model weights if available; otherwise use the current model.
if BEST_CKPT.exists():
    best_state = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
    # Best checkpoint stores a raw model state_dict.
    model.load_state_dict(best_state)
    print("Loaded best model weights:", BEST_CKPT)
elif LATEST_CKPT.exists():
    ckpt = torch.load(LATEST_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded latest model checkpoint:", LATEST_CKPT)
else:
    print("No saved checkpoint available yet; using current in-memory model.")
model.eval()


## 14. Visualize 10 test images total across four preprocessing methods

In [ ]:
# Neon-green labels with transparent backgrounds.
# Green outlines = every ground-truth annotation in the dataset.
# Magenta outlines = model predictions above the confidence threshold.
# Predicted confidence is displayed beside each predicted class.
# No model can guarantee zero missed detections; compare GT and predictions to identify misses.

def show_test_image(ds, idx, method, image_number):
    img, target = ds[idx]
    with torch.no_grad():
        pred = model([img.to(DEVICE)])[0]

    arr = img.detach().cpu().permute(1, 2, 0).numpy().clip(0, 1)
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(arr)
    ax.set_title(f"Test image {image_number}/10 | {method.upper()}")
    ax.axis("off")

    # Draw ALL ground-truth boxes; no confidence filter is applied to annotations.
    for box, label in zip(target["boxes"].numpy(), target["labels"].numpy()):
        x1, y1, x2, y2 = box
        semantic_id = int(label) - 1
        name = SEMANTIC_NAMES.get(semantic_id, "Unknown")
        ax.add_patch(plt.Rectangle(
            (x1, y1), x2-x1, y2-y1, fill=False,
            edgecolor="#39FF14", linewidth=2.5
        ))
        ax.text(
            x1, max(0, y1-5), f"GT {semantic_id}: {name}",
            color="#39FF14", fontsize=9, fontweight="bold",
            bbox=dict(facecolor="none", edgecolor="none", alpha=0)
        )

    # Draw predictions with confidence.
    pred_boxes = pred["boxes"].detach().cpu().numpy()
    pred_labels = pred["labels"].detach().cpu().numpy()
    pred_scores = pred["scores"].detach().cpu().numpy()
    for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
        if score < CONFIDENCE_THRESHOLD:
            continue
        x1, y1, x2, y2 = box
        semantic_id = int(label) - 1
        name = SEMANTIC_NAMES.get(semantic_id, "Unknown")
        ax.add_patch(plt.Rectangle(
            (x1, y1), x2-x1, y2-y1, fill=False,
            edgecolor="#FF00FF", linewidth=2
        ))
        ax.text(
            x1, min(y2, y1+14),
            f"Pred {semantic_id}: {name} | confidence {score:.2f}",
            color="#39FF14", fontsize=9, fontweight="bold",
            bbox=dict(facecolor="none", edgecolor="none", alpha=0)
        )

    plt.tight_layout()
    plt.show()
    plt.close(fig)

if TEST_DIR is None:
    print("No test split found. Check the dataset folders in Section 6.")
else:
    chosen = []
    per_method = [3, 3, 2, 2]
    for method, quota in zip(METHODS, per_method):
        ds = BHDDataset(TEST_DIR, training=False, method=method)
        ids = random.sample(range(len(ds)), min(quota, len(ds)))
        chosen.extend((method, ds, idx) for idx in ids)

    # If some methods have fewer images, fill to 10 from any available method.
    while len(chosen) < min(10, len(BHDDataset(TEST_DIR, training=False, method="original"))):
        method = random.choice(METHODS)
        ds = BHDDataset(TEST_DIR, training=False, method=method)
        idx = random.randrange(len(ds))
        chosen.append((method, ds, idx))

    for i, (method, ds, idx) in enumerate(chosen[:10], start=1):
        show_test_image(ds, idx, method, i)


## 15. Notes on bounding boxes and confidence

- **Ground-truth boxes** come from the dataset's annotation files. This notebook draws every valid annotated box on each displayed test image, using neon green.
- **Predicted boxes** are model outputs and cannot be guaranteed to match every ground-truth box. They are drawn in magenta, with neon-green class/confidence text.
- `CONFIDENCE_THRESHOLD = 0.30` controls which predictions are shown. Lowering it can reveal more low-confidence predictions but can also add false positives.
- To assess missed boxes quantitatively, calculate recall / average precision on the full test split. A visualization alone cannot guarantee zero misses.
- Check that the Roboflow source class order really is `0 = land`, `1 = water` before training.
- Checkpoints are saved under `MyDrive/BHD_batch10_preprocessing_experiment/checkpoints` after every epoch. To resume after a reset, set `RESUME = True` in Section 10 and run the setup/model/optimizer/scheduler cells before the training cell.
